## Import necessary modules
Run this cell before running any other cells

In [2]:
%load_ext autoreload
%autoreload 2

from ble import get_ble_controller
from base_ble import LOG
from cmd_types import CMD
import time
import numpy as np

LOG.propagate = False

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Printing and Logging
## Printing
You can use the **print()** function in Python to print messages to the screen. <br>
The message can be a string, or any other object, the object will be converted into a string before it is written to the screen. <br>

## Logging
You could use the logging module that is setup in *utils.py*. <br>
It prints to both your screen (standard output) as well as to log files (*ble.log*) in the *logs* directory. <br>
This is the recommended way to output messages, since the log files can help with debugging. <br>
The logging module also provides different log levels as shown below, each formatted with a different color for increased visibility. <br>

__**NOTE**__: You may notice that the DEBUG message is not printed to the screen but is printed in the log file. This is because the logging level for the screen is set to INFO and for the file is set to DEBUG. You can change the default log levels in *utils.py* (**STREAM_LOG_LEVEL** and **FILE_LOG_LEVEL**). 

## Formatting output
To format your strings, you may use %-formatting, str.format() or f-strings. <br>
The most "pythonic" way would be to use f-strings. [Here](https://realpython.com/python-f-strings/) is a good tutorial on f-strings. <br>

In [4]:
LOG.debug("debug")
LOG.info("info")
LOG.warning("warning")
LOG.error("error")
LOG.critical("critical")

2026-01-28 22:15:20,451 | INFO     |: info
2026-01-28 22:15:20,452 | WARNING  |: warning
2026-01-28 22:15:20,453 | ERROR    |: error
2026-01-28 22:15:20,453 | CRITICAL |: critical


<hr>

# BLE
## ArtemisBLEController
The class **ArtemisBLEController** (defined in *ble.py*) provides member functions to handle various BLE operations to send and receive data to/from the Artemis board, provided the accompanying Arduino sketch is running on the Artemis board. <br>

<table align="left">
     <tr>
        <th style="text-align: left; font-size: medium">Member Functions</th>
        <th style="text-align: left; font-size: medium">Description</th style="text-align: left">
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">reload_config()</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Reload changes made in <em>connection.yaml.</em></span></th style="text-align: left">
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">connect()</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Connect to the Artemis board, whose MAC address is specified in <em>connection.yaml</em>.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">disconnect()</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Disconnect from the Artemis board.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">is_connected()</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Return a boolean indicating whether your controller is connected to the Artemis board or not.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">send_command(cmd_type, data)</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Send the command <strong>cmd_type</strong> (integer) with <strong>data</strong> (string) to the Artemis board.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">receive_float(uuid) <br> receive_string(uuid) <br> receive_int(uuid)</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Read the GATT characteristic (specified by its <strong>uuid</strong>) of type float, string or int. <br> The type of the GATT
            characteristic is determined by the classes BLEFloatCharacteristic, BLECStringCharacteristic or
            BLEIntCharacteristic in the Arduino sketch.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">start_notify(uuid, notification_handler)</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Activate notifications on the GATT characteristic (specified by its <strong>uuid</strong>). <br> <strong>notification_handler</strong> is a
            function callback which must accept two inputs; the first will be a uuid string object and the second will
            be the bytearray of the characteristic value.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">bytearray_to_float(byte_array) <br> bytearray_to_string(byte_array) <br> bytearray_to_int(byte_array)</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Convert the <strong>bytearray</strong> to float, string or int, respectively. <br> You may use these functions inside your
            notification callback function.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">stop_notify(uuid)</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Stop notifications on the GATT characteristic (specified by its <strong>uuid</strong>).</span></th>
    </tr>
</table>

<table align="left">
     <tr>
        <th style="text-align: left; font-size: medium">Member Variables</th>
        <th style="text-align: left; font-size: medium">Description</th style="text-align: left">
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">uuid</span></th>
        <th style="text-align: left"><span style="font-weight: normal">A dictionary that stores the UUIDs of the various characteristics specified in <em>connection.yaml</em>.</span></th>
    </tr>
</table>

## Configuration
- The MAC address, Service UUID and GATT characteristic UUIDs are defined in the file: *connection.yaml*.
- They should match the UUIDs used in the Arduino sketch.
- The artemis board running the base code should display its MAC address in the serial monitor.
- Update the **artemis_address** in *connection.yaml*, accordingly.
- Make sure to call **ble.reload_config()** or **get_ble_controller()** (which internally calls **reload_config()**) after making any changes to your configuration file.

<hr>

In the below cell, we create an **ArtemisBLEController** object using **get_ble_controller()** (defined in *ble.py*), which creates and/or returns a single instance of **ArtemisBLEController**. <br>
<span style="color:rgb(240,50,50)"> __NOTE__: Do not use the class directly to instantiate an object. </span><br>

In [15]:
# Get ArtemisBLEController object
ble = get_ble_controller()

# Connect to the Artemis Device
ble.connect()

2026-01-28 22:19:32,837 | INFO     |: Looking for Artemis Nano Peripheral Device: c0:81:31:25:23:64
2026-01-28 22:19:32,839 | INFO     |: Scanning for device with address: c0:81:31:25:23:64, service UUID: d1e59283-ea64-46d2-9619-feda9179e362


2026-01-28 22:19:42,933 | INFO     |: Found 1 device(s) advertising service d1e59283-ea64-46d2-9619-feda9179e362
2026-01-28 22:19:42,934 | INFO     |: Selecting device: CA4CC09F-B046-D9C9-C03F-F04C7AD18338 (name: Artemis BLE)
2026-01-28 22:19:44,073 | INFO     |: Connected to c0:81:31:25:23:64


## Receive data from the Artemis board

The cell below shows examples of reading different types (as defined in the Arduino sketch) of GATT characteristics.

In [8]:
# Read a float GATT Charactersistic
f = ble.receive_float(ble.uuid['RX_FLOAT'])
print(f)

3.5


In [9]:
# Read a string GATT Charactersistic
s = ble.receive_string(ble.uuid['RX_STRING'])
print(s)

[->9.000<-]


## Send a command to the Artemis board
Send the PING command and read the reply string from the string characteristic RX_STRING. <br>
__NOTE__: The **send_command()** essentially sends a string data to the GATT characteristic (TX_CMD_STRING). The GATT characteristic in the Arduino sketch is of type BLECStringCharacteristic.

In [10]:
ble.send_command(CMD.PING, "")

In [11]:
s = ble.receive_string(ble.uuid['RX_STRING'])
print(s)

PONG


The cell below shows an example of the SEND_TWO_INTS command. <br> The two values in the **data** are separated by a delimiter "|". <br>
Refer Lab 2 documentation for more information on the command protocol.

In [12]:
ble.send_command(CMD.SEND_TWO_INTS, "2|-6")

The Artemis board should print the two integers to the serial monitor in the ArduinoIDE. 

## Disconnect

In [13]:
# Disconnect
ble.disconnect()

# Lab Tasks

## Task 1: ECHO Command

In [16]:
ble.send_command(CMD.ECHO, "HiHello")
time.sleep(0.5)
s = ble.receive_string(ble.uuid['RX_STRING'])
print(f"Received: {s}")

Received: Robot says -> HiHello :)


## Task 2: SEND_THREE_FLOATS Command

In [17]:
# 1.5, 2.7, 3.14
ble.send_command(CMD.SEND_THREE_FLOATS, "1.5|2.7|3.14")

## Task 3: GET_TIME_MILLIS Command

In [18]:
ble.send_command(CMD.GET_TIME_MILLIS, "")
time.sleep(0.5)
s = ble.receive_string(ble.uuid['RX_STRING'])
print(f"Received: {s}")

Received: T:105609


## Task 4 & 5: Notification Handler and Real-time Data Transfer

In [19]:
timestamps = []
arrival_times = []

def notification_handler(uuid, byte_array):
    global timestamps, arrival_times
    s = ble.bytearray_to_string(byte_array)
    arrival_time = time.time()
    
    if s.startswith("T:"):
        try:
            timestamp = int(s.split(":")[1].split("|")[0])
            timestamps.append(timestamp)
            arrival_times.append(arrival_time)
            print(f"Received timestamp: {timestamp} ms")
        except:
            print(f"Error parsing: {s}")

ble.start_notify(ble.uuid['RX_STRING'], notification_handler)
print("Ready to receive data.")

Ready to receive data.


In [20]:
timestamps.clear()
arrival_times.clear()

print("Requesting timestamps for 5 seconds...")
start_time = time.time()
count = 0

while time.time() - start_time < 5:
    ble.send_command(CMD.GET_TIME_MILLIS, "")
    count += 1
    time.sleep(0.1)  # 100ms

print(f"\nSent {count} requests")
time.sleep(1)
print(f"Received {len(timestamps)} timestamps")

if len(arrival_times) >= 2:
    duration = arrival_times[-1] - arrival_times[0]
    rate = len(timestamps) / duration
    print(f"\nEffective data transfer rate: {rate:.2f} messages/second")
    print(f"Average interval: {duration/len(timestamps)*1000:.2f} ms between messages")

Requesting timestamps for 5 seconds...
Received timestamp: 114104 ms
Received timestamp: 114249 ms
Received timestamp: 114405 ms
Received timestamp: 114550 ms
Received timestamp: 114705 ms
Received timestamp: 114852 ms
Received timestamp: 115000 ms
Received timestamp: 115154 ms
Received timestamp: 115296 ms
Received timestamp: 115449 ms
Received timestamp: 115601 ms
Received timestamp: 115749 ms
Received timestamp: 115902 ms
Received timestamp: 116048 ms
Received timestamp: 116196 ms
Received timestamp: 116344 ms
Received timestamp: 116498 ms
Received timestamp: 116648 ms
Received timestamp: 116795 ms
Received timestamp: 116946 ms
Received timestamp: 117098 ms
Received timestamp: 117273 ms
Received timestamp: 117427 ms
Received timestamp: 117575 ms
Received timestamp: 117728 ms
Received timestamp: 117873 ms
Received timestamp: 118023 ms
Received timestamp: 118176 ms
Received timestamp: 118348 ms
Received timestamp: 118505 ms
Received timestamp: 118651 ms
Received timestamp: 118798 ms



Received timestamp: 118950 ms


In [24]:
print("Starting data recording on Artemis...")
ble.send_command(CMD.START_RECORDING, "")
time.sleep(0.5)
s = ble.receive_string(ble.uuid['RX_STRING'])
print(f"Response: {s}")

print("Recording for 3 seconds...")
time.sleep(3)

print("Stopping data recording...")
ble.send_command(CMD.STOP_RECORDING, "")
time.sleep(0.5)
s = ble.receive_string(ble.uuid['RX_STRING'])
print(f"Response: {s}")

Starting data recording on Artemis...
Response: Recording started
Recording for 3 seconds...
Stopping data recording...
Response: Recording stopped. Samples: 344


## Task 6: Array Storage and Batch Data Transfer

In [22]:
timestamps.clear()
arrival_times.clear()

print("Requesting stored timestamp data...")
ble.send_command(CMD.SEND_TIME_DATA, "")

time.sleep(5)

print(f"\nReceived {len(timestamps)} timestamps")
if len(timestamps) > 0:
    print(f"First timestamp: {timestamps[0]} ms")
    print(f"Last timestamp: {timestamps[-1]} ms")
    if len(timestamps) > 1:
        time_diff = timestamps[-1] - timestamps[0]
        sampling_rate = (len(timestamps) - 1) / (time_diff / 1000.0)
        print(f"Sampling rate: {sampling_rate:.2f} samples/second")
        print(f"Average interval: {time_diff/(len(timestamps)-1):.2f} ms")

Requesting stored timestamp data...

Received 0 timestamps


Received timestamp: 129187 ms
Received timestamp: 129198 ms
Received timestamp: 129211 ms
Received timestamp: 129221 ms
Received timestamp: 129231 ms
Received timestamp: 129241 ms
Received timestamp: 129251 ms
Received timestamp: 129261 ms
Received timestamp: 129274 ms
Received timestamp: 129285 ms
Received timestamp: 129295 ms
Received timestamp: 129305 ms
Received timestamp: 129315 ms
Received timestamp: 129329 ms
Received timestamp: 129339 ms
Received timestamp: 129349 ms
Received timestamp: 129359 ms
Received timestamp: 129369 ms
Received timestamp: 129381 ms
Received timestamp: 129391 ms
Received timestamp: 129403 ms
Received timestamp: 129413 ms
Received timestamp: 129423 ms
Received timestamp: 129433 ms
Received timestamp: 129443 ms
Received timestamp: 129453 ms
Received timestamp: 129463 ms
Received timestamp: 129473 ms
Received timestamp: 129483 ms
Received timestamp: 129493 ms
Received timestamp: 129507 ms
Received timestamp: 129517 ms
Received timestamp: 129527 ms
Received t

## Task 7: Temperature Readings with Timestamps

In [ ]:
timestamps.clear()
temp_readings = []

def temp_notification_handler(uuid, byte_array):
    """Notification handler for temperature readings"""
    global timestamps, temp_readings
    s = ble.bytearray_to_string(byte_array)
    
    # T:123456|C:25.5
    if "T:" in s and "|C:" in s:
        try:
            parts = s.split("|")
            timestamp = int(parts[0].split(":")[1])
            temp = float(parts[1].split(":")[1])
            timestamps.append(timestamp)
            temp_readings.append(temp)
            print(f"Time: {timestamp} ms, Temp: {temp:.2f} °C")
        except Exception as e:
            print(f"Error parsing: {s}, Error: {e}")

ble.stop_notify(ble.uuid['RX_STRING'])
ble.start_notify(ble.uuid['RX_STRING'], temp_notification_handler)

print("Requesting temperature readings...")
ble.send_command(CMD.GET_TEMP_READINGS, "")

time.sleep(5)

print(f"\nReceived {len(temp_readings)} temperature readings")
if len(temp_readings) > 0:
    print(f"Min temp: {min(temp_readings):.2f} °C")
    print(f"Max temp: {max(temp_readings):.2f} °C")
    print(f"Avg temp: {np.mean(temp_readings):.2f} °C")

Requesting temperature readings...

Received 0 temperature readings


Time: 159670 ms, Temp: 25.25 °C
Time: 159680 ms, Temp: 25.25 °C
Time: 159690 ms, Temp: 24.66 °C
Time: 159700 ms, Temp: 25.25 °C
Time: 159711 ms, Temp: 25.83 °C
Time: 159721 ms, Temp: 25.25 °C
Time: 159731 ms, Temp: 25.25 °C
Time: 159741 ms, Temp: 25.25 °C
Time: 159755 ms, Temp: 25.25 °C
Time: 159765 ms, Temp: 24.66 °C
Time: 159775 ms, Temp: 25.25 °C
Time: 159785 ms, Temp: 24.66 °C
Time: 159795 ms, Temp: 25.25 °C
Time: 159805 ms, Temp: 25.25 °C
Time: 159815 ms, Temp: 25.25 °C
Time: 159825 ms, Temp: 25.25 °C
Time: 159836 ms, Temp: 25.25 °C
Time: 159846 ms, Temp: 25.25 °C
Time: 159856 ms, Temp: 25.25 °C
Time: 159866 ms, Temp: 25.25 °C
Time: 159876 ms, Temp: 25.25 °C
Time: 159886 ms, Temp: 24.66 °C
Time: 159898 ms, Temp: 25.25 °C
Time: 159908 ms, Temp: 24.66 °C
Time: 159918 ms, Temp: 25.25 °C
Time: 159928 ms, Temp: 25.83 °C
Time: 159939 ms, Temp: 25.25 °C
Time: 159949 ms, Temp: 25.25 °C
Time: 159959 ms, Temp: 25.83 °C
Time: 159969 ms, Temp: 24.66 °C
Time: 159979 ms, Temp: 25.25 °C
Time: 15

In [26]:
ble.stop_notify(ble.uuid['RX_STRING'])
ble.disconnect()

In [ ]:
import sys
sys.path.append("/Users/qiandaoliu/fastrobot/ble_robot_1.4/ble_arduino/tests/imu_test")
from imu_analysis import *

df = serial_collect(port="/dev/cu.usbserial-210", duration_s=10)


SerialException: [Errno 2] could not open port /dev/cu.usbmodem14101: [Errno 2] No such file or directory: '/dev/cu.usbmodem14101'